# 1. Introduction to Pandas

**Pandas** is the core library for data manipulation in Python. In this notebook, we will learn:
- How to create and inspect DataFrames
- Selection, filtering, and indexing
- Creating new columns with `apply`
- GroupBy aggregation and pivot tables
- Merging (joining) DataFrames

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

print(f"Pandas version: {pd.__version__}")

## 1.1 Creating a DataFrame

A DataFrame is a two-dimensional, tabular data structure with labeled axes (rows and columns).
You can create one from a dictionary where keys become column names.

In [ ]:
data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'Age': [25, 30, 35, 28],
    'Salary': [45000, 55000, 70000, 52000],
    'City': ['Paris', 'Lyon', 'Paris', 'Marseille']
}
df = pd.DataFrame(data)
print(df)
print(f"\nShape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")

## 1.2 Loading a Real Dataset

Seaborn provides built-in datasets. The **Tips** dataset records restaurant tips with information about the bill, tip amount, and customer demographics.

In [ ]:
tips = sns.load_dataset('tips')
print(f"Tips shape: {tips.shape}")
tips.head(5)

## 1.3 Selection and Filtering

Pandas supports:
- **Column selection**: `df['col']` or `df[['col1', 'col2']]`
- **Boolean filtering**: `df[df['col'] > value]`
- **`.loc`**: label-based indexing with both row and column selectors

In [ ]:
# Filter tips greater than $5
big_tips = tips[tips['tip'] > 5]
print(f"Tips > $5: {len(big_tips)} rows")
print(big_tips.head(3))

# Combine conditions with loc
mask = (tips['sex'] == 'Female') & (tips['day'] == 'Sun')
print("\nFemales on Sunday:")
tips.loc[mask, ['total_bill', 'tip']].head(3)

## 1.4 Creating New Columns

You can create derived columns using vectorized operations or `apply` with a lambda function.

In [ ]:
tips['tip_pct'] = (tips['tip'] / tips['total_bill'] * 100).round(1)

tips['bill_category'] = tips['total_bill'].apply(
    lambda x: 'High' if x > 30 else ('Medium' if x > 15 else 'Low'))

print(tips[['total_bill', 'tip', 'tip_pct', 'bill_category']].head(5))
print(f"\nBill category counts:\n{tips['bill_category'].value_counts()}")

## 1.5 GroupBy Aggregation

**GroupBy** splits the data into groups, applies a function, and combines the results.
This is the pandas equivalent of SQL's `GROUP BY`.

In [ ]:
print("Mean by day:")
print(tips.groupby('day')[['total_bill', 'tip']].mean().round(2))

print("\nMultiple aggregations:")
agg = tips.groupby(['day', 'time']).agg(
    n_meals=('total_bill', 'count'),
    avg_bill=('total_bill', 'mean'),
    avg_tip=('tip', 'mean')
).round(2)
print(agg)

## 1.6 Pivot Tables and Merges

- **Pivot tables** reshape data for cross-tabulation analysis
- **Merge** combines two DataFrames on a shared key (like SQL JOINs)

In [ ]:
# Pivot table
pivot = tips.pivot_table(values='tip', index='day', columns='sex',
                         aggfunc='mean').round(2)
print("Pivot table (mean tip by day and sex):")
print(pivot)

# Merge example
clients = pd.DataFrame({'client_id': [1, 2, 3, 4],
                         'name': ['Alice', 'Bob', 'Charlie', 'Diana']})
orders = pd.DataFrame({'order_id': [101, 102, 103, 104],
                        'client_id': [1, 2, 2, 5],
                        'amount': [150, 200, 80, 300]})

print("\nInner join:")
print(pd.merge(clients, orders, on='client_id', how='inner'))
print("\nLeft join:")
print(pd.merge(clients, orders, on='client_id', how='left'))

## Key Takeaways

| Operation | Method |
|---|---|
| Create DataFrame | `pd.DataFrame(dict)` |
| Filter rows | `df[df['col'] > val]` |
| New column | `df['new'] = expression` |
| Group and aggregate | `df.groupby('col').agg(...)` |
| Pivot table | `df.pivot_table(...)` |
| Merge | `pd.merge(df1, df2, on='key')` |